In [1]:
import pandas as pd
from datasets import Dataset
import re

df = pd.read_csv('data/train.csv')
drop_df = df.drop(['location'], axis=1)
drop_df['keyword'] = drop_df['keyword'].fillna('unknown')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


drop_df['text'] = drop_df['text'].apply(clean_text)
drop_df['text'] = "keyword: " + drop_df['keyword'].apply(clean_text) + ", text: " + drop_df['text']
drop_df = drop_df.drop(['keyword'], axis=1)

dataset = Dataset.from_pandas(drop_df)

In [2]:
from transformers import BertTokenizerFast,BertForSequenceClassification
from transformers import TrainingArguments,Trainer

model_path = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model_path)

#分词
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    
#map可在样本上批量执行操作
tokenized_dataset = dataset.map(tokenize_function, batched=True)

def transform_dataset(tokenized_dataset):
    tokenized_dataset = tokenized_dataset.remove_columns(["id"])
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("target", "labels")
    return tokenized_dataset

new_dataset = transform_dataset(tokenized_dataset)
split_dataset = new_dataset.train_test_split(test_size=0.1)

train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

Map:   0%|          | 0/7613 [00:00<?, ? examples/s]

In [3]:
import numpy as np
model = BertForSequenceClassification.from_pretrained(model_path, num_labels=2)

import evaluate
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    x,y = eval_pred
    pred = np.argmax(x, axis=1)
    return metric.compute(predictions=pred, references=y)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# 设置训练参数
training_args = TrainingArguments(
    output_dir="model_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,              # 略微提高基础学习率 (2e-5 是 BERT 最经典的调优参数)
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # 3 轮足够了
    weight_decay=0.01,               # 将 0.1 降到 0.01，0.1 对基础 BERT 来说惩罚有点太重
    warmup_ratio=0.1,                # 【新增】前 10% 的步数用来缓慢增加学习率，防止初期震荡破坏预训练权重
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",# 【新增】明确告诉模型：在 epoch 结束时，保存 Accuracy 最高的那个，而不是 Loss 最低的
    greater_is_better=True           # 【新增】配合上一行，数值越大越好
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.410585,0.830709
2,0.455054,0.429609,0.830709
3,0.313856,0.501954,0.829396


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1287, training_loss=0.3564773180175402, metrics={'train_runtime': 90.2676, 'train_samples_per_second': 227.69, 'train_steps_per_second': 14.258, 'total_flos': 1351930380203520.0, 'train_loss': 0.3564773180175402, 'epoch': 3.0})

In [5]:
test_df = pd.read_csv('data/test.csv')
test_ids = test_df['id'] 

test_df = test_df.drop(['location'], axis=1)
test_df['keyword'] = test_df['keyword'].fillna('unknown')
test_df['text'] = test_df['text'].apply(clean_text)
test_df['text'] = "keyword: " + test_df['keyword'].apply(clean_text) + ", text: " + test_df['text']
test_df = test_df.drop(['keyword'], axis=1)

test_dataset = Dataset.from_pandas(test_df)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

predictions = trainer.predict(tokenized_test_dataset)

import numpy as np
pred = np.argmax(predictions.predictions, axis=1)

submission = pd.DataFrame({'id': test_ids, 'target': pred})
submission.to_csv('outputs/submission_bert.csv', index=False)
print("Submission saved successfully!")

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

Submission saved successfully!
